# Linear Baselines

This notebook runs ridge and CCA baselines against the current repo loaders.
It respects the active modality split definitions from `config.yaml`.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))


            import numpy as np
            import pandas as pd
            from sklearn.cross_decomposition import CCA
            from sklearn.linear_model import Ridge

            from src.eval_utils import create_dataset, load_clip_cache, load_config

            CONFIG_PATH = ROOT / "config.yaml"
            MODALITY = "eeg"  # eeg, meg, fmri
            SUBJECT = 1

            config = load_config(CONFIG_PATH)
            clip_dict = load_clip_cache(config)


In [ ]:
def flatten_dataset(dataset):
    xs = []
    ys = []
    image_ids = []
    for index in range(len(dataset)):
        item = dataset[index]
        xs.append(item["x"].numpy().reshape(-1))
        ys.append(item["y_clip"].numpy())
        image_ids.append(item["image_id"])
    return np.stack(xs), np.stack(ys), image_ids

def retrieval_scores(predictions, candidate_ids, candidate_embeddings, image_ids):
    grouped = {}
    for image_id, prediction in zip(image_ids, predictions):
        grouped.setdefault(image_id, []).append(prediction)

    candidate_norms = np.linalg.norm(candidate_embeddings, axis=1, keepdims=True)
    candidate_matrix = candidate_embeddings / np.clip(candidate_norms, 1e-12, None)

    top1 = 0
    top5 = 0
    total = 0
    for image_id, vectors in grouped.items():
        averaged = np.mean(np.stack(vectors, axis=0), axis=0, dtype=np.float32)
        averaged = averaged / max(np.linalg.norm(averaged), 1e-12)
        scores = averaged @ candidate_matrix.T
        ranking = np.argsort(scores)[::-1]
        top_ids = [candidate_ids[idx] for idx in ranking[:5]]
        top1 += int(top_ids[0] == image_id)
        top5 += int(image_id in top_ids)
        total += 1

    return {
        "top1": 100.0 * top1 / total,
        "top5": 100.0 * top5 / total,
        "candidate_images": total,
    }

train_dataset = create_dataset(config, MODALITY, "train", subject=SUBJECT, quiet=True)
test_dataset = create_dataset(config, MODALITY, "test", subject=SUBJECT, quiet=True)

X_train, Y_train, _ = flatten_dataset(train_dataset)
X_test, Y_test, test_ids = flatten_dataset(test_dataset)
candidate_ids = sorted(set(test_ids))
candidate_embeddings = np.stack([clip_dict[image_id] for image_id in candidate_ids], axis=0)


In [ ]:
ridge = Ridge(alpha=10000.0)
ridge.fit(X_train, Y_train)
ridge_predictions = ridge.predict(X_test)
ridge_scores = retrieval_scores(ridge_predictions, candidate_ids, candidate_embeddings, test_ids)

cca = CCA(n_components=min(100, Y_train.shape[1], X_train.shape[1]))
cca.fit(X_train, Y_train)
X_test_c, _ = cca.transform(X_test, Y_test)
_, candidate_c = cca.transform(np.zeros((len(candidate_ids), X_train.shape[1])), candidate_embeddings)
cca_scores = retrieval_scores(X_test_c, candidate_ids, candidate_c, test_ids)

pd.DataFrame(
    [
        {"baseline": "ridge", **ridge_scores},
        {"baseline": "cca", **cca_scores},
    ]
)
